# ✈️ AI Trip Planning Agent
> Built with OpenAI GPT-4o-mini · Gradio Chat UI · Session Memory

**Features:**
- 🗺️ `get_places` — fetches top tourist spots
- 💰 `estimate_budget` — estimates trip cost in INR
- 🗓️ `generate_itinerary` — builds a day-by-day plan
- 🧠 **Memory** — remembers your trip details across the conversation
- 🔄 **Modify** — update any detail (days, budget style, destination) mid-chat

In [1]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q openai gradio

In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os
import re
from openai import OpenAI

In [ ]:
# ── Cell 3: API Key ───────────────────────────────────────────────────────────
# Paste your OpenAI API key below
os.environ["OPENAI_API_KEY"] = "sk-proj--Kq8..."
client = OpenAI()

In [4]:
# ── Cell 4: Memory ────────────────────────────────────────────────────────────
# Stores the user's trip details across conversation turns
memory = {
    "destination": None,
    "days": None,
    "preference": None,   # "budget", "luxury", or "normal"
    "places": [],
    "itinerary": None,
    "budget": None,
}

def reset_memory():
    """Wipe memory when the user starts a new trip."""
    for k in memory:
        memory[k] = [] if k == "places" else None

print("✅ Memory initialised")

✅ Memory initialised


In [5]:
# ── Cell 5: Extraction helpers ────────────────────────────────────────────────

def extract_destination(text):
    """
    Pulls destination from patterns like:
      'trip to Paris', 'go to New York for', 'visit Goa'
    """
    text_lower = text.lower()
    patterns = [
        r"(?:trip to|travel to|go to|visit|plan.*?to|heading to)\s+([a-zA-Z][a-zA-Z ]{1,30}?)(?:\s+for|\s+in|\d|$|[,.])",
        r"^([a-zA-Z][a-zA-Z ]{1,25}?)(?:\s+for\s+\d|\s+trip)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text_lower)
        if match:
            candidate = match.group(1).strip()
            # filter out generic words
            stop_words = {"a", "an", "the", "my", "me", "us", "budget", "luxury", "normal"}
            if candidate not in stop_words and len(candidate) > 1:
                return candidate.title()
    return None


def extract_days(text):
    """Extracts number of days from '5 days', '5-day', 'five days'."""
    word_map = {"one":1,"two":2,"three":3,"four":4,"five":5,
                "six":6,"seven":7,"eight":8,"nine":9,"ten":10}
    # digit form
    m = re.search(r"(\d+)\s*[-]?\s*day", text.lower())
    if m:
        return int(m.group(1))
    # word form
    for word, num in word_map.items():
        if re.search(rf"\b{word}\b.{{0,8}}day", text.lower()):
            return num
    return None


def extract_preference(text):
    """Detects budget style from user message."""
    text_lower = text.lower()
    if any(w in text_lower for w in ["budget", "cheap", "affordable", "backpack", "low cost"]):
        return "budget"
    if any(w in text_lower for w in ["luxury", "5 star", "five star", "premium", "lavish"]):
        return "luxury"
    return None


def detect_modification(text):
    """
    Detects if user is modifying an existing plan rather than starting fresh.
    Returns (field, value) or (None, None).
    """
    text_lower = text.lower()
    # Changing days
    m = re.search(r"(?:change|make it|update|extend|reduce).{0,15}(\d+)\s*day", text_lower)
    if m:
        return "days", int(m.group(1))
    # Changing preference
    if re.search(r"(?:change|make it|switch).{0,15}(budget|luxury|normal)", text_lower):
        pref = re.search(r"(budget|luxury|normal)", text_lower)
        if pref:
            return "preference", pref.group(1)
    # Changing destination
    m = re.search(r"(?:change|go to|switch to)\s+([a-zA-Z][a-zA-Z ]{1,25})", text_lower)
    if m:
        return "destination", m.group(1).strip().title()
    return None, None


def is_new_trip(text):
    """Returns True if the user seems to be asking about a brand new trip."""
    keywords = ["plan a trip", "i want to go", "travel to", "trip to", "visit", "vacation in"]
    return any(k in text.lower() for k in keywords)


print("✅ Extraction helpers ready")

✅ Extraction helpers ready


In [6]:
# ── Cell 6: Tool — get_places ─────────────────────────────────────────────────

def get_places(destination):
    """
    Returns a list of 5 top tourist places for the given destination.
    Uses GPT-4o-mini with strict output formatting.
    """
    prompt = (
        f"List exactly 5 famous tourist places in {destination}.\n"
        "Rules:\n"
        "- Only place names, no descriptions\n"
        "- One per line\n"
        "- No numbering, no bullets, no extra text\n"
    )
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=100,
    )
    raw = res.choices[0].message.content.strip().split("\n")
    places = []
    for p in raw:
        clean = re.sub(r"^[\d\.\-\*\)]+\s*", "", p).strip()
        if clean:
            places.append(clean)
    return places[:5]


print("✅ get_places ready")

✅ get_places ready


In [7]:
# ── Cell 7: Tool — estimate_budget ────────────────────────────────────────────

def estimate_budget(destination, days, preference):
    """
    Returns an estimated trip cost in INR as a formatted string.
    Breaks down into accommodation, food, transport, activities.
    """
    style_map = {"budget": "budget (hostels, local food)",
                 "luxury": "luxury (5-star hotels, fine dining)",
                 "normal": "mid-range (3-star hotels, restaurants)"}
    style = style_map.get(preference, style_map["normal"])

    prompt = (
        f"Estimate the total trip cost in INR for {days} days in {destination} "
        f"with {style} travel style.\n"
        "Return ONLY a JSON object with these keys (values as integers, no commas or symbols):\n"
        '{"accommodation": 0, "food": 0, "transport": 0, "activities": 0, "total": 0}\n'
        "No explanation, no extra text."
    )
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=120,
    )
    raw = res.choices[0].message.content.strip()

    # Parse JSON safely
    import json
    try:
        # strip markdown code fences if present
        raw = re.sub(r"```[a-z]*", "", raw).strip()
        data = json.loads(raw)
        lines = [
            f"  🏨 Accommodation : ₹{data.get('accommodation', 0):,}",
            f"  🍽️  Food           : ₹{data.get('food', 0):,}",
            f"  🚌 Transport      : ₹{data.get('transport', 0):,}",
            f"  🎟️  Activities      : ₹{data.get('activities', 0):,}",
            f"  ─────────────────────────────",
            f"  💰 TOTAL          : ₹{data.get('total', 0):,}",
        ]
        return "\n".join(lines)
    except Exception:
        # Fallback: extract any number
        nums = re.findall(r"\d{4,}", raw)
        total = nums[-1] if nums else "N/A"
        return f"  💰 Estimated Total: ₹{total}"


print("✅ estimate_budget ready")

✅ estimate_budget ready


In [8]:
# ── Cell 8: Tool — generate_itinerary ────────────────────────────────────────

def generate_itinerary(destination, days, places, preference="normal"):
    """
    Generates a detailed day-by-day itinerary.
    Validates output and retries once if incomplete.
    """
    places_str = ", ".join(places) if places else destination
    style_hint = {
        "budget": "Focus on free/cheap activities, street food, and public transport.",
        "luxury": "Include premium experiences, fine dining, and private transfers.",
        "normal": "Mix of popular sights, local restaurants, and standard transport."
    }.get(preference, "")

    prompt = (
        f"Create a {days}-day trip itinerary for {destination}.\n"
        f"Must include these places: {places_str}.\n"
        f"{style_hint}\n\n"
        "STRICT FORMAT — follow exactly:\n"
        f"Day 1: [morning activity] → [afternoon activity] → [evening plan]\n"
        f"Day 2: ...\n"
        f"(continue up to Day {days})\n\n"
        "Rules:\n"
        f"- Include ALL {days} days, no skipping\n"
        "- Each day on its own line starting with 'Day N:'\n"
        "- Max 20 words per day\n"
        "- No extra text before or after"
    )

    def call():
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=400,
        )
        return r.choices[0].message.content.strip()

    result = call()

    # Validate: count lines starting with "Day"
    day_lines = [l for l in result.split("\n") if re.match(r"^Day \d+", l, re.IGNORECASE)]
    if len(day_lines) < days:
        result = call()   # one retry

    return result


print("✅ generate_itinerary ready")

✅ generate_itinerary ready


In [9]:
# ── Cell 9: Agent — Trip Planner ──────────────────────────────────────────────

def trip_planner_agent(user_input):
    """
    Main agent. Handles:
    - New trip planning
    - Modifying existing plan (change days / budget style / destination)
    - Follow-up questions using memory
    """
    text = user_input.strip()

    # ── Reset command ─────────────────────────────────────────────────────────
    if any(w in text.lower() for w in ["reset", "start over", "new trip", "clear"]):
        reset_memory()
        return "🔄 Memory cleared! Tell me about your next trip."

    # ── Check for modification intent first (if a plan already exists) ────────
    if memory["destination"]:
        mod_field, mod_value = detect_modification(text)
        if mod_field:
            memory[mod_field] = mod_value
            # Regenerate the affected parts
            destination = memory["destination"]
            days = memory["days"] or 3
            preference = memory["preference"] or "normal"
            places = memory["places"] or get_places(destination)
            memory["places"] = places

            change_msg = {
                "days": f"📅 Updated to {mod_value} days!",
                "preference": f"💼 Switched to {mod_value} style!",
                "destination": f"📍 Changed destination to {mod_value}!",
            }.get(mod_field, "✅ Plan updated!")

            itinerary = generate_itinerary(destination, days, places, preference)
            budget = estimate_budget(destination, days, preference)
            memory["itinerary"] = itinerary
            memory["budget"] = budget

            return format_response(destination, days, preference, places, itinerary, budget, header=change_msg)

    # ── Extract new trip details ───────────────────────────────────────────────
    dest = extract_destination(text)
    days_val = extract_days(text)
    pref_val = extract_preference(text)

    # New destination wipes the old plan
    if dest and dest != memory["destination"]:
        reset_memory()
        memory["destination"] = dest

    if days_val:
        memory["days"] = days_val
    if pref_val:
        memory["preference"] = pref_val

    # Apply defaults
    if memory["days"] is None:
        memory["days"] = 3
    if memory["preference"] is None:
        memory["preference"] = "normal"

    # Still no destination?
    if not memory["destination"]:
        return (
            "👋 Hi! I'm your AI Trip Planner.\n\n"
            "Tell me where you'd like to go! For example:\n"
            "• _'Plan a trip to Paris for 5 days'_\n"
            "• _'I want to visit Goa on a budget for 4 days'_\n"
            "• _'Luxury trip to Dubai for 7 days'_"
        )

    destination = memory["destination"]
    days = memory["days"]
    preference = memory["preference"]

    # ── Call tools ────────────────────────────────────────────────────────────
    places = get_places(destination)
    memory["places"] = places

    itinerary = generate_itinerary(destination, days, places, preference)
    memory["itinerary"] = itinerary

    budget = estimate_budget(destination, days, preference)
    memory["budget"] = budget

    return format_response(destination, days, preference, places, itinerary, budget)


def format_response(destination, days, preference, places, itinerary, budget, header=None):
    """Formats the complete trip plan as a clean readable string."""
    style_emoji = {"budget": "🎒", "luxury": "💎", "normal": "🧳"}.get(preference, "🧳")
    places_fmt = "\n".join(f"  • {p}" for p in places)
    itinerary_fmt = "\n".join(f"  {line}" for line in itinerary.split("\n") if line.strip())

    lines = []
    if header:
        lines += [header, ""]
    lines += [
        f"╔══════════════════════════════════════╗",
        f"   ✈️  TRIP PLAN  —  {destination.upper()}",
        f"╚══════════════════════════════════════╝",
        f"",
        f"📍 Destination : {destination}",
        f"🗓️  Duration    : {days} days",
        f"{style_emoji} Travel Style : {preference.capitalize()}",
        f"",
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        f"📌 TOP PLACES TO VISIT",
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        places_fmt,
        f"",
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        f"🗓️  DAY-BY-DAY ITINERARY",
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        itinerary_fmt,
        f"",
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        f"💰 BUDGET ESTIMATE ({preference.capitalize()} Style)",
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        budget,
        f"",
        f"─────────────────────────────────────",
        f"💡 You can say:",
        f"  • 'Change to 7 days' to update duration",
        f"  • 'Switch to luxury' to change style",
        f"  • 'Plan a trip to [new destination]' for a new plan",
        f"  • 'Reset' to start fresh",
    ]
    return "\n".join(lines)


print("✅ Trip Planner Agent ready")

✅ Trip Planner Agent ready


In [ ]:
# ── Cell 10: Gradio Chat UI ───────────────────────────────────────────────────
import gradio as gr

WELCOME = (
    "👋 **Welcome to your AI Trip Planner!**\n\n"
    "Just tell me where you want to go. Examples:\n"
    "- `Plan a trip to Paris for 5 days`\n"
    "- `Budget trip to Goa for 4 days`\n"
    "- `Luxury vacation in Dubai for 7 days`\n\n"
    "I'll fetch top places, build your itinerary, and estimate your budget. "
    "You can then ask me to modify anything!"
)

def chat_fn(user_message, history):
    if not user_message.strip():
        return "", history
    response = trip_planner_agent(user_message)
    history = history + [(user_message, response)]
    return "", history

def clear_fn():
    reset_memory()
    return [], []

# ── Build UI ──────────────────────────────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Trip Planner",
    css="""
        #header { text-align: center; margin-bottom: 10px; }
        #chatbox { font-family: monospace; }
        .tip-box { background: #f0f7ff; padding: 10px; border-radius: 8px;
                   border-left: 4px solid #4A90E2; font-size: 13px; }
    """
) as demo:

    # Header
    gr.Markdown(
        """
        <div id="header">
            <h1>✈️ AI Trip Planning Agent</h1>
            <p style="color:#666">Powered by GPT-4o-mini · CrewAI-style Tool Architecture · Session Memory</p>
        </div>
        """
    )

    # Tip bar
    gr.HTML(
        '<div class="tip-box">'
        '💡 <b>Tips:</b> Say <i>"change to 7 days"</i>, <i>"switch to luxury"</i>, '
        'or <i>"plan a trip to [new place]"</i> to modify your plan on the fly. '
        'Type <i>"reset"</i> to start fresh.'
        '</div>'
    )

    # Chatbot
    chatbot = gr.Chatbot(
        value=[(None, WELCOME)],
        elem_id="chatbox",
        height=520,
        bubble_full_width=False,
        label="Trip Planner"
    )

    # Input row
    with gr.Row():
        msg = gr.Textbox(
            placeholder="e.g. Plan a budget trip to Goa for 4 days",
            scale=5,
            show_label=False,
            container=False,
        )
        send_btn = gr.Button("Send ✈️", variant="primary", scale=1)

    # Memory display panel
    with gr.Accordion("🧠 Current Memory (debug)", open=False):
        mem_display = gr.JSON(label="Agent Memory", value=memory)

    # Buttons row
    with gr.Row():
        clear_btn = gr.Button("🗑️ Clear Chat & Reset", variant="secondary")
        refresh_mem = gr.Button("🔄 Refresh Memory View", variant="secondary")

    # ── Event bindings ────────────────────────────────────────────────────────
    send_btn.click(
        fn=chat_fn,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )
    msg.submit(
        fn=chat_fn,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )
    clear_btn.click(
        fn=clear_fn,
        outputs=[chatbot, chatbot]
    )
    refresh_mem.click(
        fn=lambda: memory,
        outputs=[mem_display]
    )

# ── Launch ────────────────────────────────────────────────────────────────────
demo.launch(debug=True)

/tmp/ipykernel_4200/2247753577.py:26: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_4200/2247753577.py:26: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_4200/2247753577.py:57: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_4200/2247753577.py:57: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_4200/224

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 07:49:10 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>